### Setup für alle Aufgaben

In [1]:
from pymongo import MongoClient
from datetime import datetime
from bson import ObjectId

# Verbindung zur lokalen MongoDB
client = MongoClient('localhost', 27017) 

# Datenbank "university" auswählen
db = client['university']

# Collections
colStudents = db["studenten"]
colKurse = db["kurse"]
colOrte = db["orte"]

# Vor jeder Aufgabe ggf. alte Daten löschen (optional)
colStudents.delete_many({})
colKurse.delete_many({})
colOrte.delete_many({})

DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)

#### Aufgabe 1: einfacher Insert

In [2]:
colStudents.insert_one({"name": "Anna", "alter": 22, "eingeschrieben": True})

InsertOneResult(ObjectId('69d8f71f5e620f39d5d61376'), acknowledged=True)

#### Aufgabe 2: Mehrere Dokumente einfügen

In [3]:
colStudents.insert_many([
    {"name": "Jonas", "skills": ["Python", "MongoDB"], "alter": 24},
    {"name": "Lea", "skills": ["C++", "ML"], "alter": 26}
]
)

InsertManyResult([ObjectId('69d8f71f5e620f39d5d61377'), ObjectId('69d8f71f5e620f39d5d61378')], acknowledged=True)

#### Aufgabe 3: Dokumente abfragen

In [4]:
cursor = colStudents.find({"alter": {"$gt": 2}}, {"name": 1, "alter": 1})
for doc in cursor:
    print(doc)
cursor = colStudents.find({}, {"name": 1, "alter": 1})
for doc in cursor:
    print(doc)

{'_id': ObjectId('69d8f71f5e620f39d5d61376'), 'name': 'Anna', 'alter': 22}
{'_id': ObjectId('69d8f71f5e620f39d5d61377'), 'name': 'Jonas', 'alter': 24}
{'_id': ObjectId('69d8f71f5e620f39d5d61378'), 'name': 'Lea', 'alter': 26}
{'_id': ObjectId('69d8f71f5e620f39d5d61376'), 'name': 'Anna', 'alter': 22}
{'_id': ObjectId('69d8f71f5e620f39d5d61377'), 'name': 'Jonas', 'alter': 24}
{'_id': ObjectId('69d8f71f5e620f39d5d61378'), 'name': 'Lea', 'alter': 26}


#### Aufgabe 4: Dokumente ändern

In [5]:
colStudents.update_many({"alter": {"$gt": 25}}, {"$set": {"eingeschrieben": False}})

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

#### Aufgabe 5: Löschen eines Dokuments

In [6]:
colStudents.delete_many({"name": "Jonas"})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

#### Aufgabe 6: Datumsfelder

In [7]:
colStudents.update_many({}, {"$set": {"eingeschrieben_am": datetime.utcnow()}})

UpdateResult({'n': 2, 'nModified': 2, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

#### Aufgabe 7: Sortierung und Projektion

In [8]:
cursor = colStudents.find({}).sort({"alter": -1})
for doc in cursor:
    print(doc)

{'_id': ObjectId('69d8f71f5e620f39d5d61378'), 'name': 'Lea', 'skills': ['C++', 'ML'], 'alter': 26, 'eingeschrieben': False, 'eingeschrieben_am': datetime.datetime(2026, 4, 10, 13, 11, 59, 795000)}
{'_id': ObjectId('69d8f71f5e620f39d5d61376'), 'name': 'Anna', 'alter': 22, 'eingeschrieben': True, 'eingeschrieben_am': datetime.datetime(2026, 4, 10, 13, 11, 59, 795000)}


#### Aufgabe 8: Komplexes Dokument mit eingebetteten Objekten

In [9]:
colKurse.insert_many([{
    "titel": "MongoDB Basics",
    "dozent": {"name": "Dr. Müller", "email": "mueller@example.com"},
    "studenten": [
        {"name": "Anna", "note": 1.7},
        {"name": "Lea", "note": 2.3}
    ]
}])

InsertManyResult([ObjectId('69d8f71f5e620f39d5d61379')], acknowledged=True)

#### Aufgabe 9: Suche in Arrays

In [10]:
cursor = colKurse.find({"studenten.note": {"$lt": 2.0}})
for doc in cursor:
    print(doc)

{'_id': ObjectId('69d8f71f5e620f39d5d61379'), 'titel': 'MongoDB Basics', 'dozent': {'name': 'Dr. Müller', 'email': 'mueller@example.com'}, 'studenten': [{'name': 'Anna', 'note': 1.7}, {'name': 'Lea', 'note': 2.3}]}


#### Aufgabe 10: Zählen und Gruppieren

In [22]:
cursor = colStudents.aggregate([{
    "$group": {
	    "_id": "$eingeschrieben", # Group key
		"count": {
            "$sum": 1
        }
	}
}])
for doc in cursor:
    print(doc)

{'_id': False, 'count': 1}
{'_id': True, 'count': 1}


#### Aufgabe 11: Arbeiten mit ObjectId

In [28]:
doc = colStudents.find_one()
id = doc["_id"]
print(id)

print(colStudents.find_one({"_id": ObjectId(id)}))

69d8f71f5e620f39d5d61376
{'_id': ObjectId('69d8f71f5e620f39d5d61376'), 'name': 'Anna', 'alter': 22, 'eingeschrieben': True, 'eingeschrieben_am': datetime.datetime(2026, 4, 10, 13, 11, 59, 795000)}


#### Aufgabe 12: Null-Werte & Existenzprüfung

In [35]:
#colStudents.insert_one({"name": "Tom", "phone": None})
cursor = colStudents.find({"$or": [{"phone": None}, {"phone": {"$exists": False}}]})
for doc in cursor:
    print(doc)

{'_id': ObjectId('69d8f71f5e620f39d5d61376'), 'name': 'Anna', 'alter': 22, 'eingeschrieben': True, 'eingeschrieben_am': datetime.datetime(2026, 4, 10, 13, 11, 59, 795000)}
{'_id': ObjectId('69d8f71f5e620f39d5d61378'), 'name': 'Lea', 'skills': ['C++', 'ML'], 'alter': 26, 'eingeschrieben': False, 'eingeschrieben_am': datetime.datetime(2026, 4, 10, 13, 11, 59, 795000)}
{'_id': ObjectId('69d8fcf65e620f39d5d6137a'), 'name': 'Tom', 'phone': None}


#### Aufgabe 13: Lookup

In [36]:
colStudents.delete_many({})
colNoten = db["noten"]
colNoten.delete_many({})

s1 = colStudents.insert_one({"name": "Anna"}).inserted_id
s2 = colStudents.insert_one({"name": "Lea"}).inserted_id

colNoten.insert_many([
    {"student_id": s1, "kurs": "MongoDB"},
    {"student_id": s1, "kurs": "Python"},
    {"student_id": s2, "kurs": "C++"}
])

InsertManyResult([ObjectId('69d8fedf5e620f39d5d6137d'), ObjectId('69d8fedf5e620f39d5d6137e'), ObjectId('69d8fedf5e620f39d5d6137f')], acknowledged=True)

In [ ]:
pipeline = [{
    "$lookup": {
        "from": "noten",
        "localField": "_id",
        "foreignField": "student_id",
        "as": "student_kurse"
    }
}]
for doc in colStudents.aggregate(pipeline):
    kurse = doc["student_kurse"]
    for k in kurse:
        print(k["kurs"])
    print(doc["name"])

MongoDB
Python
Anna
C++
Lea


#### Expertenaufgabe 14: GeoJSON und Geospatial Query